In [ ]:
# Rebuilt one-click Colab (direct prompt defaults)
# Stable startup with GPU preflight + CUDA torch check + URL fallbacks.

!pip install -q pygit2==1.15.1

import importlib
import os
import re
import subprocess
import sys
from IPython.display import HTML, display

REPO_URL = "https://github.com/sunsideaspect/foocus_new.git"
BRANCH = "cursor/git-eaf0"  # switch to "main" after merge if needed
REPO_DIR = "/content/Fooocus"


def has_nvidia_gpu() -> bool:
    r = subprocess.run(["bash", "-lc", "nvidia-smi -L"], capture_output=True, text=True)
    return r.returncode == 0


def torch_cuda_ok() -> bool:
    try:
        import torch
        return torch.version.cuda is not None and torch.cuda.is_available()
    except Exception:
        return False


if not has_nvidia_gpu():
    raise RuntimeError(
        "No NVIDIA GPU runtime detected. In Colab: Runtime -> Change runtime type -> GPU (T4/L4/A100), then rerun."
    )

# Keep repo/models between reruns in same session for speed.
os.chdir('/content')
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, "Fooocus"], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

# Ensure CUDA-enabled PyTorch is available (fixes "Torch not compiled with CUDA enabled").
if not torch_cuda_ok():
    print("[Setup] Installing CUDA-enabled PyTorch...")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"], check=False)
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "--upgrade",
            "torch", "torchvision", "torchaudio",
            "--index-url", "https://download.pytorch.org/whl/cu124"
        ],
        check=True,
    )
    importlib.invalidate_caches()

import torch
print(f"[Torch] version={torch.__version__}, cuda={torch.version.cuda}, is_available={torch.cuda.is_available()}")
if not torch_cuda_ok():
    raise RuntimeError(
        "CUDA torch is still unavailable. Reconnect runtime with GPU and rerun this cell."
    )

os.chdir(REPO_DIR)
# Keep launcher from trying incompatible pinned torch versions on newer Colab Python.
os.environ["TORCH_COMMAND"] = "pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124"


def get_total_vram_mb() -> int:
    r = subprocess.run(
        ["bash", "-lc", "nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -n1"],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        return 0
    try:
        return int(r.stdout.strip().splitlines()[0])
    except Exception:
        return 0


vram_mb = get_total_vram_mb()
vram_flag = "--always-high-vram" if vram_mb >= 20000 else "--always-normal-vram"
print(f"[VRAM] total={vram_mb} MB -> using {vram_flag}")

cmd = [
    sys.executable,
    "entry_with_update.py",
    "--preset", "realistic_direct_prompt",
    "--share",
    vram_flag,
    "--disable-in-browser"
]
print("Launching:", " ".join(cmd))

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

share_re = re.compile(r"https://[a-z0-9-]+\.gradio\.live")
local_re = re.compile(r"Running on local URL:\s+(http://127\.0\.0\.1:(\d+))")
printed_public = False
printed_proxy = False

for line in proc.stdout:
    print(line, end="")

    m = share_re.search(line)
    if m and not printed_public:
        share_url = m.group(0)
        print("\n✅ Gradio public URL:\n" + share_url + "\n")
        display(HTML(f'<a href="{share_url}" target="_blank">Open Gradio public URL</a>'))
        printed_public = True

    lm = local_re.search(line)
    if lm and not printed_proxy:
        port = lm.group(2)
        try:
            from google.colab import output
            proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({port})")
            if proxy_url:
                print("\n✅ Colab proxy fallback URL:\n" + proxy_url + "\n")
                display(HTML(f'<a href="{proxy_url}" target="_blank">Open Colab proxy URL (fallback)</a>'))
                printed_proxy = True
        except Exception as e:
            print(f"[Info] Could not create Colab proxy URL: {e}")
